3. Inférence FinBERT      → scores de sentiment, distributions

---
## Imports initiaux

In [1]:
import pandas as pd
import os

# Logging
import logging
logger = logging.getLogger(__name__)

---
## Modeling

In [ ]:
from transformers import pipeline

from huggingface_hub import login
login(token=os.getenv("TOKEN_HF"))

pipe = pipeline("text-classification", model="ProsusAI/finbert")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [3]:
# Engine
from pipeline.core.src.sql import get_engine

engine = get_engine()

[INFO] pipeline_sql: Engine created — neondb_owner@ep-quiet-sun-al5bcj46-pooler.c-3.eu-central-1.aws.neon.tech/theguardian-sentiment


In [4]:
# Label unprocessed
from pipeline.core.src.sql import fetch_unprocessed

articles = fetch_unprocessed(engine=engine, schema="theguardian")
articles.shape

[INFO] pipeline_sql: Fetched 34122 unprocessed articles from theguardian.articles


(34122, 2)

In [5]:
sample_articles = articles.iloc[:2]
sample_articles

,id,text
0,business/2017/jan/01/robin-hoods-sherwood-fore...,Robin Hood's Sherwood Forest faces fracking th...
1,business/2017/jan/01/gambling-firms-charmed-mp...,Gambling firms charmed MPs ahead of betting re...


In [6]:
results = [pipe(text, truncation=True, max_length=512)[0] for text in sample_articles['text']]
sample_articles['sentiment_label'] = [r['label'] for r in results]
sample_articles['sentiment_score'] = [r['score'] for r in results]
sample_articles

,id,text,sentiment_label,sentiment_score
0,business/2017/jan/01/robin-hoods-sherwood-fore...,Robin Hood's Sherwood Forest faces fracking th...,neutral,0.622462
1,business/2017/jan/01/gambling-firms-charmed-mp...,Gambling firms charmed MPs ahead of betting re...,negative,0.497815


In [7]:
records = sample_articles[["id", "sentiment_label", "sentiment_score"]].rename(
    columns={"sentiment_label": "label", "sentiment_score": "score"}
).to_dict(orient="records")
records

[{'id': 'business/2017/jan/01/robin-hoods-sherwood-forest-faces-fracking-threat',
  'label': 'neutral',
  'score': 0.6224623322486877},
 {'id': 'business/2017/jan/01/gambling-firms-charmed-mps-ahead-of-betting-review-in-2016',
  'label': 'negative',
  'score': 0.49781525135040283}]

---
# Test insertion

In [8]:
from pipeline.core.src.sql import update_sentiment_batch


update_sentiment_batch(engine=engine, records=records, schema="theguardian")

[INFO] pipeline_sql: Updated sentiment for 2 articles in theguardian.articles
